In [ ]:
# -*- coding: utf-8 -*-


FinAgent - Data Cleaning & Processing Module
Course: IT Application in Banking and Finance

Design decisions:
- Forward-fill (ffill) + backward-fill (bfill): standard for financial time series
  to handle missing trading days (weekends, holidays)
- Simple daily returns (pct_change): chosen for interpretability and visualization
- Volatility window=30: industry standard for monthly volatility estimation
- Outlier threshold=3 std from mean: flags statistically anomalous returns


In [ ]:

import pandas as pd
import numpy as np
import os



============================================================
STEP 1: LOAD RAW DATA
============================================================


In [ ]:

stock = pd.read_csv(r"C:\Users\LENOVO\Downloads\stock2_prices.csv")
macro = pd.read_csv(r"C:\Users\LENOVO\Downloads\macro2_prices.csv")

# Remove duplicate header rows if any
stock = stock[stock.iloc[:, 0] != 'Date']
macro = macro[macro.iloc[:, 0] != 'Date']



============================================================
STEP 2: RENAME COLUMNS
============================================================


In [ ]:

stock.columns = [
    'Date',
    'AAPL_Close', 'AAPL_AdjClose', 'AAPL_Volume',
    'AMZN_Close', 'AMZN_AdjClose', 'AMZN_Volume',
    'JNJ_Close',  'JNJ_AdjClose',  'JNJ_Volume',
    'JPM_Close',  'JPM_AdjClose',  'JPM_Volume',
    'META_Close', 'META_AdjClose', 'META_Volume',
    'MSFT_Close', 'MSFT_AdjClose', 'MSFT_Volume',
    'NVDA_Close', 'NVDA_AdjClose', 'NVDA_Volume',
    'TSLA_Close', 'TSLA_AdjClose', 'TSLA_Volume',
]

macro.columns = [
    'Date',
    'SP500_Close', 'SP500_AdjClose', 'SP500_Volume',
    'Gold_Close',  'Gold_AdjClose',  'Gold_Volume',
    'Oil_Close',   'Oil_AdjClose',   'Oil_Volume',
]



============================================================
STEP 3: DATA TYPE NORMALIZATION
============================================================


In [ ]:

stock['Date'] = pd.to_datetime(stock['Date'], errors='coerce', dayfirst=False)
macro['Date'] = pd.to_datetime(macro['Date'], errors='coerce', dayfirst=False)

stock.dropna(subset=['Date'], inplace=True)
macro.dropna(subset=['Date'], inplace=True)

for col in stock.columns[1:]:
    stock[col] = pd.to_numeric(stock[col], errors='coerce')

for col in macro.columns[1:]:
    macro[col] = pd.to_numeric(macro[col], errors='coerce')



============================================================
STEP 4: MERGE STOCK + MACRO
============================================================


In [ ]:

data = pd.merge(stock, macro, on='Date', how='left')
data.sort_values('Date', inplace=True)
data.set_index('Date', inplace=True)



============================================================
STEP 5: HANDLE MISSING VALUES
============================================================


In [ ]:

data.ffill(inplace=True)
data.bfill(inplace=True)
print(f"Missing values after fill: {data.isnull().sum().sum()}")



============================================================
STEP 6: REMOVE DUPLICATES
============================================================


In [ ]:

# Handle duplicates by index (Date) — more precise than row-level dedup
before = len(data)
data = data[~data.index.duplicated(keep='first')]
print(f"Removed {before - len(data)} duplicate rows")



============================================================
STEP 7: FEATURE ENGINEERING — STOCKS
============================================================


In [ ]:

close_cols = [
    'AAPL_Close', 'AMZN_Close', 'JNJ_Close', 'JPM_Close',
    'META_Close', 'MSFT_Close', 'NVDA_Close', 'TSLA_Close'
]

for col in close_cols:
    name = col.replace('_Close', '')
    data[f'{name}_return']     = data[col].pct_change()
    data[f'{name}_MA7']        = data[col].rolling(window=7).mean()
    data[f'{name}_MA30']       = data[col].rolling(window=30).mean()
    data[f'{name}_volatility'] = data[f'{name}_return'].rolling(window=30).std()



============================================================
STEP 8: FEATURE ENGINEERING — MACRO
============================================================


In [ ]:

macro_assets = ['SP500', 'Gold', 'Oil']

for asset in macro_assets:
    col = f'{asset}_Close'
    data[f'{asset}_return']     = data[col].pct_change()
    data[f'{asset}_MA7']        = data[col].rolling(window=7).mean()
    data[f'{asset}_MA30']       = data[col].rolling(window=30).mean()
    data[f'{asset}_volatility'] = data[f'{asset}_return'].rolling(window=30).std()



============================================================
STEP 9: DROP NaN
============================================================


In [ ]:

data.dropna(inplace=True)



============================================================
STEP 10: OUTLIER DETECTION
============================================================


In [ ]:

all_assets = [col.replace('_Close', '') for col in close_cols] + macro_assets

for name in all_assets:
    ret_col = f'{name}_return'
    mean = data[ret_col].mean()
    std  = data[ret_col].std()
    data[f'{name}_is_outlier'] = (data[ret_col] - mean).abs() > 3 * std
    outlier_count = data[f'{name}_is_outlier'].sum()
    print(f"{name}: {outlier_count} outliers detected")



============================================================
STEP 11: FINALIZE & EXPORT
============================================================


In [ ]:

data.reset_index(inplace=True)

print("\n===================================")
print("DATA CLEANING COMPLETED")
print("===================================")
print(f"\nDataset Shape: {data.shape}")
print(f"\nFirst 5 Rows:\n{data.head()}")

output_folder = r"d:\Documents\IBFM"
os.makedirs(output_folder, exist_ok=True)

csv_path  = os.path.join(output_folder, "cleaned_data.csv")
xlsx_path = os.path.join(output_folder, "cleaned_data.xlsx")

data.to_csv(csv_path, index=False)
data.to_excel(xlsx_path, index=False)

print(f"\nSaved CSV:  {csv_path}")
print(f"Saved XLSX: {xlsx_path}")
